# Modelos generativos de texto

Veremos passo-a-passo como usar um modelo generativo de texto do Hugginface. Veremos dois tipos de modelos:

1. Modelos treinados apenas para completar textos
2. Modelos refinados para conversa com o usuário

É importante salientar que a biblioteca Hugginface possui os chamados *pipelines*, que permitem fazer o que veremos neste notebook de forma mais fácil. Mas pipelines abstraem totalmente os passos intermediários do processo.

### Modelo de geração de texto Llama

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = "cuda"

# Outros modelos possíveis:
#'meta-llama/Llama-3.2-3B'
#'meta-llama/Llama-3.1-8B'
#'meta-llama/Llama-3.1-70B'
#'meta-llama/Llama-3.1-405B'
name = "meta-llama/Llama-3.2-1B"

# O uso da biblioteca bitsandbytes permite a quantização dos pesos para reduzir uso de memória
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    name, 
    torch_dtype=torch.bfloat16,  # Precisão para criar o modelo antes de carregar os pesos
    device_map="auto",           # Carrega os pesos na GPU até o limite de memória. Se precisar 
                                 # de mais memória, o restante do modelo é carregado na CPU.
    quantization_config=quantization_config   # Estratégia de quantização
    )
tokenizer = AutoTokenizer.from_pretrained(
    name, 
    padding_side="left"         # Onde inserir o token de padding para criar batches de entrada
    )

# O modelo llama não foi treinado com um token de padding, mas ele é necessário para a inferência
# utilizando batches. Podemos usar o token de final de sentença para representar o padding. Esse
# token não é utilizado na atenção do modelo, então não deve afetar a geração de texto.
tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.eos_token_id

In [19]:
# Tokenização de duas sentenças de texto
model_inputs = tokenizer(["A list of colors: red, blue", "1, 2"], padding=True, return_tensors="pt").to(device)
model_inputs

{'input_ids': tensor([[128000,     32,   1160,    315,   8146,     25,   2579,     11,   6437],
        [128001, 128001, 128001, 128001, 128000,     16,     11,    220,     17]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1, 1]], device='cuda:0')}

In [20]:
# Decodificação dos tokens. Note que tokens extra foram adicionados para indicar o começo do texto,
# o fim do texto e o padding
tokenizer.batch_decode(model_inputs["input_ids"])

['<|begin_of_text|>A list of colors: red, blue',
 '<|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|begin_of_text|>1, 2']

In [21]:
# A primeira camada do modelo é a camada de embedding. Ela transforma cada um dos 128k tokens possíveis
# em um vetor possuindo 2048 valores. Usualmente, modelos treinados apenas no idioma inglês possuem
# em torno de 40k tokens. Modelos multilinguagem precisam de mais tokens para representar
# palavras em diferentes idiomas.
model.model.embed_tokens

Embedding(128256, 2048)

In [22]:
embeddings = model.model.embed_tokens(model_inputs["input_ids"])
embeddings.shape
# Saída possui tamanho bs x nro de tokens x d_model

torch.Size([2, 9, 2048])

In [23]:
# Geração de texto. O modelo simplesmente encontra os tokens com maior probabilidade 
# de completar os tokens de entrada
# O resultado da geração consiste em "nro tokens de entrada + max_new_tokens".
generated_ids = model.generate(**model_inputs, max_new_tokens = 20)
generated_ids

tensor([[128000,     32,   1160,    315,   8146,     25,   2579,     11,   6437,
             11,  14071,     11,   6307,     11,  19087,     11,  25977,     11,
          18718,     11,  14198,     11,   3776,     11,   4251,     11,  18004,
             11,   6437],
        [128001, 128001, 128001, 128001, 128000,     16,     11,    220,     17,
             11,    220,     18,     11,    220,     19,     11,    220,     20,
             11,    220,     21,     11,    220,     22,     11,    220,     23,
             11,    220]], device='cuda:0')

In [24]:
tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

['A list of colors: red, blue, yellow, green, orange, purple, pink, brown, black, white, gray, blue',
 '1, 2, 3, 4, 5, 6, 7, 8, ']

Como o modelo gerou novas palavras?

Para cada novo token a ser gerado, a sequência inteira é passada pelo modelo para gerar atributos para o **último token da sequência de saída**. Esses atributos são então classificados em ~128k classes através de uma camada linear. Vamos ver como o processo funciona.

In [25]:
generated_ids = model.generate(
    **model_inputs, 
    max_new_tokens = 5, 
    output_hidden_states = True,     # Retorna todas as ativações do modelo
    return_dict_in_generate = True,  
    use_cache = False                # Evita de suprimir algumas ativações na saída
    )

A saída `generated_ids["hidden_states"]` possui dimensão (nro novos tokens, nro camadas do modelo, bs, tamanho seq, d_model), ou seja, todas as ativações geradas por todas as camadas do modelo. Estamos interessados apenas nas ativações da última camada:

In [26]:
for new_token_id, activations in enumerate(generated_ids["hidden_states"]):
    last_activation = activations[-1]
    print(f"New token id: {new_token_id}, activation shape: {last_activation.shape}")

New token id: 0, activation shape: torch.Size([2, 9, 2048])
New token id: 1, activation shape: torch.Size([2, 10, 2048])
New token id: 2, activation shape: torch.Size([2, 11, 2048])
New token id: 3, activation shape: torch.Size([2, 12, 2048])
New token id: 4, activation shape: torch.Size([2, 13, 2048])


Por exemplo, para gerar o último token, a sequência de entrada possui tamanho 13 tokens. A última camada gera uma ativação de tamanho (bs, 13, d_model). Uma camada linear é então aplicada a essa ativação para classificar o token.

In [27]:
# Última camada do modelo, uma camada linear de classificação
model.lm_head

Linear(in_features=2048, out_features=128256, bias=False)

In [28]:
logits = model.lm_head(last_activation)
logists_last_token = logits[:, -1]
# Probabilidades do último token ser de cada uma das 128k classes (tokens do vocabulário)
logists_last_token.shape

torch.Size([2, 128256])

### Modelo com refinamento de instrução

Modelos *instruct* foram treinados para conversação. O modelo gera palavras que representem uma conversa natural com o usuário

In [29]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Outros modelos possíveis:
#'meta-llama/Llama-3.2-3B-Instruct'
#'meta-llama/Llama-3.1-8B-Instruct'
#'meta-llama/Llama-3.1-70B-Instruct'
#'meta-llama/Llama-3.1-405B-Instruct'
name = "meta-llama/Llama-3.2-1B-Instruct"

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    name, 
    torch_dtype=torch.bfloat16,  
    device_map="auto",           
    quantization_config=quantization_config
    )
tokenizer = AutoTokenizer.from_pretrained(
    name
    )

tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.eos_token_id

Um modelo com instrução possui um template que indica como uma conversa deve ser formatada. Tipicamente, uma conversa possui os seguintes atores:

1. system: Texto que é adicionado a **todas as entradas do usuário**. Tipicamente, esse texto envolve instruções de como o modelo deve se 'comportar' durante toda a conversa. 
2. user: Representa o usuário
3. assistant: Representa o modelo

A conversa também pode ter artefatos como ferramentas (tools) e documentos. Esses artefatos não serão vistos neste notebook.

Uma conversa é representada por um conjunto de mensagens, que são transformadas em uma única string:

In [30]:
# Exemplo de instrução de sistema e mensagem do usuário
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {
        "role": "user", 
        "content": "How many cats does it take to change a light bulb?"
    },
]

# Conversão da conversa em uma string
model_inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=False,
    add_generation_prompt=True   # Adiciona o texto assistant ao final da string para guiar o modelo
    )
print(model_inputs)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 30 Mar 2025

You are a friendly chatbot who always responds in the style of a pirate<|eot_id|><|start_header_id|>user<|end_header_id|>

How many cats does it take to change a light bulb?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




A string gerada acima possui um cabeçalho (system) com instruções para guiar o modelo, incluindo a data que o modelo foi treinado e a data atual, para que o modelo possa responder adequadamente perguntas que ultrapassem a data de treinamento.

A lista de mensagens é transformada em uma string através de um template Jinja:

In [31]:
print(tokenizer.chat_template)

{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- 

In [32]:
# Criação dos tokens para entrada no modelo
model_inputs = tokenizer.apply_chat_template(
    messages, 
    add_generation_prompt=True, 
    return_tensors="pt",
    return_dict=True,
    )
model_inputs

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    966,   2947,    220,   2366,     20,    271,   2675,    527,
            264,  11919,   6369,   6465,    889,   2744,  31680,    304,    279,
           1742,    315,    264,  55066, 128009, 128006,    882, 128007,    271,
           4438,   1690,  19987,   1587,    433,   1935,    311,   2349,    264,
           3177,  46912,     30, 128009, 128006,  78191, 128007,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [33]:
generated_ids = model.generate(**model_inputs.to(device), do_sample=True, max_new_tokens=50)
generated_ids

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    966,   2947,    220,   2366,     20,    271,   2675,    527,
            264,  11919,   6369,   6465,    889,   2744,  31680,    304,    279,
           1742,    315,    264,  55066, 128009, 128006,    882, 128007,    271,
           4438,   1690,  19987,   1587,    433,   1935,    311,   2349,    264,
           3177,  46912,     30, 128009, 128006,  78191, 128007,    271,   9014,
             81,     11,  20043,   4363,     75,  92986,      0, 115518,   2610,
            258,      6,    264,    436,   3390,     11,  30276,     88,      0,
          51849,    649,    956,   2349,   3177,  54320,     11,  64828,     30,
           2435,   2351,    810,   8173,    304,    308,    680,    258,      6,
            304,    279,   7160,   1395,   4214,    323,    523,  51410,      6,
           1306,  21120,  28

A saída do modelo inclui a entrada e os `max_new_tokens` gerados. Para obter apenas a saída, decodificamos apenas os novos tokens gerados:

In [34]:
input_length = model_inputs["input_ids"].shape[1]
generated_ids[:, input_length:]
print(tokenizer.decode(generated_ids[0, input_length:]))

Arrr, ye landlubber! Yer askin' a riddle, matey! Cats can't change light bulbs, savvy? They're more interested in nappin' in the sunbeams and chasin' after laser pointers than
